In [55]:
import pandas as pd
import glob
import numpy as np

In [56]:
sec = pd.read_csv('../reports_data/BA_SEC.csv')
z_spread = pd.read_csv('../Z_spreads/Z_BA.csv')
stocks = pd.read_csv('../stock_data/BA.csv')

In [57]:
maturity= pd.Timestamp("2038-02-15")

In [58]:
sec.head(2)

,Date,Shares,STD,LTD
0,2009-07-22,726432299,416000000,8700000000
1,2009-10-21,726599064,973000000,10065000000


In [59]:
z_spread.head(5)

,Unnamed: 0,Date,Z,rf,Days_to_maturity
0,0,2025-03-31,143.135848,4.411829,4704
1,1,2025-03-27,141.817852,4.560286,4708
2,2,2025-03-26,141.091525,4.523977,4709
3,3,2025-03-24,148.729514,4.505195,4711
4,4,2025-03-21,146.204817,4.423997,4714


In [60]:
stocks.head(2)

,Date,Open,High,Low,Close,Adj Close,Volume,Dividends
0,2016-01-04,141.380005,141.699997,139.429993,140.500000,126.005096,5719500,0.0
1,2016-01-05,140.990005,141.699997,139.770004,141.070007,126.516312,3357000,0.0


In [61]:
for d in [sec, z_spread, stocks]:
    d["Date"] = pd.to_datetime(d["Date"])

In [62]:
df = pd.merge_asof(
    stocks.sort_values("Date"),
    z_spread[["Date", "Z", "rf", "Days_to_maturity"]].sort_values("Date"),
    on="Date",
    direction="backward"
)


In [63]:
df = pd.merge_asof(
    df,
    sec[["Date", "Shares", "STD", "LTD"]].sort_values("Date"),
    on="Date",
    direction="backward"
)

In [64]:
df["sigma_60"] = np.log(df["Adj Close"]).diff().rolling(60).std() * np.sqrt(252)
df["sigma_500"] = np.log(df["Adj Close"]).diff().rolling(500).std() * np.sqrt(252)
df["sigma_100"] = np.log(df["Adj Close"]).diff().rolling(100).std() * np.sqrt(252)

In [65]:
iv = []
for file in sorted(glob.glob("../Options/theta_history/BA/greeks_implied_volatility_1h/*.csv")):
    x = pd.read_csv(file, usecols=[
        "timestamp", "expiration", "strike", "bid", "ask", "implied_vol", "underlying_price"
    ])
    x = x[(x["bid"] > 0) & (x["ask"] >= x["bid"]) & (x["implied_vol"] > 0)
          & (x["underlying_price"] > 0)]
    x = x[x["timestamp"] == x["timestamp"].max()]
    x = x[(x["strike"] / x["underlying_price"] - 1).abs() <= 0.05]
    x = x[x["expiration"] == x["expiration"].max()]
    distance = (x["strike"] - x["underlying_price"]).abs()
    iv.append((
        pd.Timestamp(file.rsplit("/", 1)[-1][:10]),
        x.loc[distance == distance.min(), "implied_vol"].mean()
    ))

iv = pd.DataFrame(iv, columns=["Date", "IV"])
df["IV"] = df["Date"].map(iv.set_index("Date")["IV"])

In [66]:
df

,Date,Open,High,Low,Close,Adj Close,Volume,Dividends,Z,rf,Days_to_maturity,Shares,STD,LTD,sigma_60,sigma_500,sigma_100,IV
0,2016-01-04,141.380005,141.699997,139.429993,140.500000,126.005096,5719500,0.0,NaN,NaN,NaN,669983631,614000000,8402000000,NaN,NaN,NaN,NaN
1,2016-01-05,140.990005,141.699997,139.770004,141.070007,126.516312,3357000,0.0,NaN,NaN,NaN,669983631,614000000,8402000000,NaN,NaN,NaN,NaN
2,2016-01-06,138.830002,139.350006,137.259995,138.830002,124.507401,4254100,0.0,NaN,NaN,NaN,669983631,614000000,8402000000,NaN,NaN,NaN,NaN
3,2016-01-07,136.380005,136.669998,132.509995,133.009995,119.287827,7013700,0.0,NaN,NaN,NaN,669983631,614000000,8402000000,NaN,NaN,NaN,NaN
4,2016-01-08,133.699997,135.100006,129.800003,129.990005,116.579399,5776900,0.0,NaN,NaN,NaN,669983631,614000000,8402000000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2728,2026-11-09,177.389999,177.389999,168.889999,172.380005,172.380005,11421200,0.0,143.135848,4.411829,4704.0,792449940,4569000000,41351000000,0.359743,0.360741,0.361247,0.36925
2729,2026-11-10,171.270004,174.330002,170.500000,173.380005,173.380005,6213700,0.0,143.135848,4.411829,4704.0,792449940,4569000000,41351000000,0.357933,0.360526,0.361114,0.37675
2730,2026-11-11,172.809998,174.820007,172.179993,174.539993,174.539993,5638100,0.0,143.135848,4.411829,4704.0,792449940,4569000000,41351000000,0.358201,0.360076,0.361275,0.37330
2731,2026-11-12,173.470001,175.160004,171.300003,171.550003,171.550003,6028800,0.0,143.135848,4.411829,4704.0,792449940,4569000000,41351000000,0.359219,0.359341,0.361103,0.37695


In [67]:
df.to_csv('BA_for_CG.csv')